# Percolation Threshold Extractive Summarization

This notebook demonstrates the **vocabulary percolation threshold** method for extractive summarization and evaluates it against fixed-ratio TF baselines on CNN/DailyMail documents.

**Key idea:** Build a vocabulary co-occurrence graph from the article. Greedily select the highest-scoring sentences until the Giant Connected Component (GCC) of the accumulated vocabulary reaches a fraction `theta` of the full article's GCC. This `theta` is the *percolation threshold* that controls compression.

**Evaluation:** Compare 4 theta values (0.6/0.7/0.8/0.9) against fixed-ratio TF baselines (10%/20%/30%) using ROUGE recall metrics, Wilcoxon signed-rank tests (Holm-Bonferroni corrected), compression ratio analysis, and network feature regression.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# rouge-score and loguru are NOT pre-installed on Colab
_pip('rouge-score==0.1.2')
_pip('loguru==0.7.3')

# Core packages: pre-installed on Colab, install locally to match Colab env
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'scikit-learn==1.6.1', 'statsmodels==0.14.6',
         'networkx==3.6.1', 'nltk==3.9.1', 'matplotlib==3.10.0')

In [ ]:
import json
import math
import random
import sys
from collections import Counter

import networkx as nx
import nltk
import numpy as np
import matplotlib.pyplot as plt
from rouge_score import rouge_scorer
from scipy import stats
from statsmodels.stats.multitest import multipletests
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-6a9498-vocabulary-percolation-threshold-for-sel/main/round-1/evaluation-1/demo/mini_demo_data.json"

import json, os

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(f"Loaded {len(data)} pre-computed document results")
print(f"Fields per doc: {list(data[0].keys())}")

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# All tunable parameters. Start small; scale up for a full run.

THETAS = [0.6, 0.7, 0.8, 0.9]       # percolation thresholds to evaluate
FIXED_RATIOS = [0.10, 0.20, 0.30]   # fixed-ratio baselines
N_DOCS = 30                          # number of documents to use from loaded data
                                     # (original: 2000 from CNN/DailyMail test set)
RANDOM_SEED = 42

## NLTK Setup

Download punkt and stopwords if not already present.

In [ ]:
def ensure_nltk():
    import os
    nltk_root = "/root/nltk_data"
    if nltk_root not in nltk.data.path:
        nltk.data.path.insert(0, nltk_root)
    for pkg, subdir in [("punkt_tab", "tokenizers/punkt_tab"), ("stopwords", "corpora/stopwords")]:
        full_path = os.path.join(nltk_root, subdir)
        if not os.path.isdir(full_path):
            try:
                nltk.download(pkg, quiet=True)
            except Exception:
                pass

ensure_nltk()
from nltk.corpus import stopwords
stop_words = set(stopwords.words("english"))
scorer_obj = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
print(f"Stop words loaded: {len(stop_words)}, ROUGE scorer ready")

## Text Processing Functions

These functions:
1. Tokenize and filter words (removing stopwords and short tokens)
2. Build a vocabulary co-occurrence graph where nodes=words, edges=co-occurrence in the same sentence
3. Measure the Giant Connected Component (GCC) size — the key structural signal

In [ ]:
def preprocess(text: str, stop_words: set) -> list:
    try:
        tokens = nltk.word_tokenize(text.lower())
    except Exception:
        tokens = text.lower().split()
    return [w for w in tokens if w.isalpha() and len(w) >= 2 and w not in stop_words]


def build_vocab_graph(sentences_words: list) -> nx.Graph:
    G = nx.Graph()
    for words in sentences_words:
        unique = list(set(words))
        G.add_nodes_from(unique)
        for i in range(len(unique)):
            for j in range(i + 1, len(unique)):
                if G.has_edge(unique[i], unique[j]):
                    G[unique[i]][unique[j]]["weight"] += 1
                else:
                    G.add_edge(unique[i], unique[j], weight=1)
    return G


def gcc_size(G: nx.Graph) -> int:
    if len(G) == 0:
        return 0
    return max((len(c) for c in nx.connected_components(G)), default=0)


def compute_tf(sentences_words: list) -> dict:
    c = Counter()
    for words in sentences_words:
        c.update(words)
    return dict(c)


def score_sentences(sentences_words: list, tf: dict) -> list:
    return [sum(tf.get(w, 0) for w in words) for words in sentences_words]

print("Text processing functions defined")

## Summarizers

**Percolation summarizer:** Ranks sentences by TF score, greedily adds them until the GCC of accumulated vocabulary reaches `theta * full_GCC`.

**Fixed-ratio baseline:** Simply takes the top `ceil(ratio * n)` sentences by TF score.

In [ ]:
def percolation_summary(sentences_words, tf, full_G, full_gcc, theta):
    n = len(sentences_words)
    if n == 0 or full_gcc == 0:
        return list(range(n)), n

    scores = score_sentences(sentences_words, tf)
    ranked = sorted(range(n), key=lambda i: scores[i], reverse=True)

    accumulated_words = set()
    selected = []
    for idx in ranked:
        selected.append(idx)
        accumulated_words.update(sentences_words[idx])
        sub = full_G.subgraph(accumulated_words)
        cur_gcc = gcc_size(sub)
        if cur_gcc / full_gcc >= theta:
            break

    k_star = len(selected)
    return selected, k_star


def fixed_ratio_summary(sentences_words, tf, ratio):
    n = len(sentences_words)
    k = max(1, math.ceil(ratio * n))
    scores = score_sentences(sentences_words, tf)
    ranked = sorted(range(n), key=lambda i: scores[i], reverse=True)
    return sorted(ranked[:k])


def evaluate_rouge(selected_indices, all_sentences, reference, scorer_obj):
    summary_text = " ".join(all_sentences[i] for i in sorted(selected_indices))
    if not summary_text.strip():
        summary_text = all_sentences[0] if all_sentences else ""
    scores = scorer_obj.score(reference, summary_text)
    return {k: {"p": v.precision, "r": v.recall, "f": v.fmeasure} for k, v in scores.items()}


def network_properties(G):
    if len(G) == 0:
        return {"avg_degree": 0.0, "clustering_coefficient": 0.0, "graph_density": 0.0,
                "num_nodes": 0, "num_edges": 0}
    degrees = [d for _, d in G.degree()]
    avg_degree = float(np.mean(degrees))
    try:
        clustering = nx.average_clustering(G)
    except Exception:
        clustering = 0.0
    density = nx.density(G)
    return {
        "avg_degree": avg_degree,
        "clustering_coefficient": clustering,
        "graph_density": density,
        "num_nodes": len(G),
        "num_edges": G.number_of_edges(),
    }

print("Summarizer functions defined")

## Use Pre-computed Results

The loaded `mini_demo_data.json` contains pre-computed per-document results (ROUGE scores, compression ratios, network properties) from the full CNN/DailyMail run. We use these directly for statistical evaluation, skipping the expensive dataset download and per-document processing.

We take `N_DOCS` documents from the loaded data.

In [ ]:
# Use the pre-computed per-document results directly
per_doc = data[:N_DOCS]
print(f"Using {len(per_doc)} documents for statistical evaluation")
print(f"Example doc keys: {list(per_doc[0].keys())}")
print(f"Percolation thetas available: {list(per_doc[0]['percolation'].keys())}")
print(f"Fixed ratios available: {list(per_doc[0]['fixed'].keys())}")

## Statistical Evaluation

Runs:
1. **ROUGE summary table** — mean/std for each method
2. **Wilcoxon signed-rank tests** — all 36 percolation×fixed×metric pairs with Holm-Bonferroni correction
3. **Compression ratio stats** — split by CNN (short) vs DailyMail (long) proxy
4. **Network feature regression** — R² predicting compression ratio from graph properties

In [ ]:
def statistical_evaluation(per_doc):
    n = len(per_doc)
    print(f"Running stats on {n} documents")

    methods_perc = [str(t) for t in THETAS]
    methods_fixed = [str(r) for r in FIXED_RATIOS]
    rouge_metrics = ["rouge1_r", "rouge2_r", "rougeL_r"]

    def get_scores(method_type, key, metric):
        return np.array([d[method_type][key].get(metric, np.nan) for d in per_doc])

    # ── Rouge summary table ──────────────────────────────────────────────────
    summary_rows = []
    for t in THETAS:
        key = str(t)
        row = {"method": f"percolation@{t}"}
        for m in ["rouge1_r", "rouge2_r", "rougeL_r", "rouge1_f", "rouge2_f", "rougeL_f"]:
            arr = get_scores("percolation", key, m)
            row[f"{m}_mean"] = float(np.mean(arr))
            row[f"{m}_std"] = float(np.std(arr))
        summary_rows.append(row)

    for r in FIXED_RATIOS:
        key = str(r)
        row = {"method": f"fixed@{int(r*100)}pct"}
        for m in ["rouge1_r", "rouge2_r", "rougeL_r", "rouge1_f", "rouge2_f", "rougeL_f"]:
            arr = get_scores("fixed", key, m)
            row[f"{m}_mean"] = float(np.mean(arr))
            row[f"{m}_std"] = float(np.std(arr))
        summary_rows.append(row)

    # ── Wilcoxon tests (36 comparisons) ─────────────────────────────────────
    wilcoxon_raw = []
    for t in THETAS:
        for r in FIXED_RATIOS:
            for metric in rouge_metrics:
                a = get_scores("percolation", str(t), metric)
                b = get_scores("fixed", str(r), metric)
                diff = a - b
                try:
                    stat, p = stats.wilcoxon(diff, alternative="greater", zero_method="wilcox")
                except ValueError:
                    stat, p = 0.0, 1.0
                wilcoxon_raw.append({
                    "method_a": f"percolation@{t}",
                    "method_b": f"fixed@{int(r*100)}pct",
                    "metric": metric,
                    "statistic": float(stat),
                    "p_raw": float(p),
                })

    p_raws = [x["p_raw"] for x in wilcoxon_raw]
    reject, p_corrected, _, _ = multipletests(p_raws, alpha=0.05, method="holm")
    wilcoxon_tests = []
    for i, x in enumerate(wilcoxon_raw):
        wilcoxon_tests.append({**x, "p_corrected": float(p_corrected[i]), "significant": bool(reject[i])})

    # ── Compression ratio stats ──────────────────────────────────────────────
    best_theta = max(THETAS, key=lambda t: np.mean(get_scores("percolation", str(t), "rouge1_r")))

    def compression_stats(docs):
        ratios = np.array([d["percolation"][str(best_theta)]["compression_ratio"] for d in docs])
        return {
            "mean": float(np.mean(ratios)),
            "std": float(np.std(ratios)),
            "min": float(np.min(ratios)),
            "max": float(np.max(ratios)),
        }

    n_sentences_list = [d["n_sentences"] for d in per_doc]
    median_sents = float(np.median(n_sentences_list))
    cnn_docs = [d for d in per_doc if d["n_sentences"] <= median_sents]
    dm_docs = [d for d in per_doc if d["n_sentences"] > median_sents]

    overall_stats = compression_stats(per_doc)
    overall_stats["std_exceeds_10pp"] = overall_stats["std"] > 0.10

    comp_ratio_stats = {
        "overall": overall_stats,
        "cnn": compression_stats(cnn_docs) if cnn_docs else {},
        "dailymail": compression_stats(dm_docs) if dm_docs else {},
        "std_exceeds_10pp": overall_stats["std"] > 0.10,
        "best_theta_used": float(best_theta),
    }

    # ── Regression: network features → compression ratio ────────────────────
    feature_names = ["avg_degree", "clustering_coefficient", "graph_density", "num_nodes", "n_sentences"]
    X_rows = []
    y_vals = []
    for d in per_doc:
        net = d["network"]
        X_rows.append([
            net["avg_degree"],
            net["clustering_coefficient"],
            net["graph_density"],
            net["num_nodes"],
            float(d["n_sentences"]),
        ])
        y_vals.append(d["percolation"][str(best_theta)]["compression_ratio"])

    X = np.array(X_rows)
    y = np.array(y_vals)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    reg = LinearRegression()
    reg.fit(X_scaled, y)
    y_pred = reg.predict(X_scaled)
    ss_res = np.sum((y - y_pred) ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0

    pearson_corr = {}
    for i, feat in enumerate(feature_names):
        p_r, p_p = stats.pearsonr(X[:, i], y)
        pearson_corr[feat] = {"r": float(p_r), "p": float(p_p)}

    regression = {
        "r2": float(r2),
        "best_theta": float(best_theta),
        "features": feature_names,
        "coefficients": [float(c) for c in reg.coef_],
        "intercept": float(reg.intercept_),
        "pearson_correlations": pearson_corr,
    }

    # ── Verdict ──────────────────────────────────────────────────────────────
    best_perc_mean = max(
        float(np.mean(get_scores("percolation", str(t), "rouge1_r"))) for t in THETAS
    )
    best_fixed_mean = max(
        float(np.mean(get_scores("fixed", str(r), "rouge1_r"))) for r in FIXED_RATIOS
    )
    best_fixed_ratio = max(
        FIXED_RATIOS, key=lambda r: float(np.mean(get_scores("fixed", str(r), "rouge1_r")))
    )

    a = get_scores("percolation", str(best_theta), "rouge1_r")
    b = get_scores("fixed", str(best_fixed_ratio), "rouge1_r")
    try:
        _, p_best = stats.wilcoxon(a - b, alternative="greater", zero_method="wilcox")
    except ValueError:
        p_best = 1.0

    rouge1_sig = any(
        wt["significant"] for wt in wilcoxon_tests if wt["metric"] == "rouge1_r"
    )

    verdict = {
        "hypothesis_supported": rouge1_sig and comp_ratio_stats["std_exceeds_10pp"],
        "best_percolation_theta": float(best_theta),
        "best_fixed_ratio": float(best_fixed_ratio),
        "best_percolation_rouge1r_mean": float(best_perc_mean),
        "best_fixed_rouge1r_mean": float(best_fixed_mean),
        "rouge1_recall_improvement": float(best_perc_mean - best_fixed_mean),
        "rouge1_significant": rouge1_sig,
        "compression_ratio_variable": comp_ratio_stats["std_exceeds_10pp"],
        "p_best_pair_wilcoxon": float(p_best),
    }

    return {
        "n_documents": n,
        "rouge_summary_table": summary_rows,
        "wilcoxon_tests": wilcoxon_tests,
        "compression_ratio_stats": comp_ratio_stats,
        "regression": regression,
        "verdict": verdict,
    }


eval_result = statistical_evaluation(per_doc)
print("Statistical evaluation complete")

## Results & Visualization

Key findings:
- ROUGE-1 recall across all methods (percolation vs fixed-ratio)
- Wilcoxon significance summary
- Compression ratio distribution
- Network feature correlations with compression ratio

In [ ]:
# ── Print verdict ────────────────────────────────────────────────────────────
v = eval_result["verdict"]
print("=" * 60)
print(f"VERDICT: hypothesis_supported={v['hypothesis_supported']}")
print(f"  Best theta={v['best_percolation_theta']}, best fixed={v['best_fixed_ratio']}")
print(f"  ROUGE-1 recall improvement: {v['rouge1_recall_improvement']:.4f}")
print(f"  Compression std > 10pp: {v['compression_ratio_variable']}")
print(f"  ROUGE-1 Wilcoxon significant: {v['rouge1_significant']}")
print(f"  Regression R²: {eval_result['regression']['r2']:.4f}")
print("=" * 60)

# ── ROUGE summary table ──────────────────────────────────────────────────────
print("\nROUGE-1 Recall Summary:")
print(f"{'Method':<22} {'ROUGE-1r':>10} {'ROUGE-2r':>10} {'ROUGE-Lr':>10}")
print("-" * 55)
for row in eval_result["rouge_summary_table"]:
    print(f"{row['method']:<22} {row['rouge1_r_mean']:>10.4f} {row['rouge2_r_mean']:>10.4f} {row['rougeL_r_mean']:>10.4f}")

# ── Wilcoxon significance count ──────────────────────────────────────────────
sig_count = sum(1 for wt in eval_result["wilcoxon_tests"] if wt["significant"])
total = len(eval_result["wilcoxon_tests"])
print(f"\nWilcoxon tests significant (Holm-corrected): {sig_count}/{total}")

# ── Plot: ROUGE-1 recall by method ──────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

methods = [row["method"] for row in eval_result["rouge_summary_table"]]
colors = ["steelblue"] * len(THETAS) + ["coral"] * len(FIXED_RATIOS)

for ax_idx, (metric_key, metric_label) in enumerate([
    ("rouge1_r", "ROUGE-1 Recall"),
    ("rouge2_r", "ROUGE-2 Recall"),
    ("rougeL_r", "ROUGE-L Recall"),
]):
    means = [row[f"{metric_key}_mean"] for row in eval_result["rouge_summary_table"]]
    stds = [row[f"{metric_key}_std"] for row in eval_result["rouge_summary_table"]]
    ax = axes[ax_idx]
    bars = ax.bar(range(len(methods)), means, color=colors, alpha=0.8, yerr=stds, capsize=4)
    ax.set_xticks(range(len(methods)))
    ax.set_xticklabels(methods, rotation=35, ha="right", fontsize=8)
    ax.set_title(metric_label)
    ax.set_ylabel("Mean Score")
    ax.set_ylim(0, 1.05)

# Add legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='steelblue', alpha=0.8, label='Percolation'),
                   Patch(facecolor='coral', alpha=0.8, label='Fixed-ratio')]
fig.legend(handles=legend_elements, loc='upper right')
fig.suptitle(f"Percolation vs Fixed-Ratio Summarization ({len(per_doc)} docs)", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# ── Plot: compression ratio distribution + network feature correlations ───────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Compression ratios for best theta
best_theta = eval_result["verdict"]["best_percolation_theta"]
comp_ratios = [d["percolation"][str(best_theta)]["compression_ratio"] for d in per_doc]
axes[0].hist(comp_ratios, bins=15, color="steelblue", alpha=0.8, edgecolor="white")
axes[0].axvline(np.mean(comp_ratios), color="red", linestyle="--", label=f"mean={np.mean(comp_ratios):.2f}")
axes[0].set_title(f"Compression Ratio Distribution (theta={best_theta})")
axes[0].set_xlabel("Compression Ratio (k*/n)")
axes[0].set_ylabel("Count")
axes[0].legend()

# Pearson correlations
pearson = eval_result["regression"]["pearson_correlations"]
feat_names = list(pearson.keys())
corr_vals = [pearson[f]["r"] for f in feat_names]
bar_colors = ["steelblue" if v >= 0 else "coral" for v in corr_vals]
axes[1].barh(feat_names, corr_vals, color=bar_colors, alpha=0.8)
axes[1].axvline(0, color="black", linewidth=0.8)
axes[1].set_title(f"Network Feature Pearson r → Compression Ratio\n(R²={eval_result['regression']['r2']:.3f})")
axes[1].set_xlabel("Pearson r")

plt.tight_layout()
plt.show()

print("\nCompression ratio stats (best theta):")
cs = eval_result["compression_ratio_stats"]["overall"]
print(f"  mean={cs['mean']:.3f}, std={cs['std']:.3f}, min={cs['min']:.3f}, max={cs['max']:.3f}")
print(f"  std > 10pp threshold: {cs['std_exceeds_10pp']}")